In [ ]:
import os
import struct
from pathlib import Path

import numpy as np


BASE_DIR = Path("anms_files_validation")  

#hw nms output
nms_kps_bin   = BASE_DIR / "seq1_img1_nms_kps.bin"
nms_count_txt = BASE_DIR / "seq1_img1_nms_count.txt"

# HW ANMS output
hw_anms_kps_bin   = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_anms_kps.bin"
hw_anms_count_txt = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_anms_count.txt"

# output to write
py_anms_kps_bin   = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_py_anms_kps.bin"
py_anms_count_txt = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_py_anms_count.txt"

# ANMS parameters
K_top   = 2048
N_final = 1000
c_anms  = 0.9

print("NMS input file:", nms_kps_bin)
print("NMS count file:", nms_count_txt)
print("HW output file:", hw_anms_kps_bin)
print("K_top =", K_top, "N_final =", N_final, "c =", c_anms)

In [ ]:
def read_int_txt(path):
    with open(path, "r") as f:
        return int(f.read().strip())

def write_int_txt(path, value):
    with open(path, "w") as f:
        f.write(f"{int(value)}\n")

def read_u64_bin(path, count=None):
    arr = np.fromfile(path, dtype=np.uint64)
    if count is not None:
        arr = arr[:int(count)]
    return arr

def write_u64_bin(path, arr_u64):
    arr_u64 = np.asarray(arr_u64, dtype=np.uint64)
    arr_u64.tofile(path)

In [ ]:
def kp_x(w):
    return np.uint16(w & np.uint64(0xFFFF))

def kp_y(w):
    return np.uint16((w >> np.uint64(16)) & np.uint64(0xFFFF))

def kp_sbits(w):
    return np.uint32((w >> np.uint64(32)) & np.uint64(0xFFFFFFFF))

def pack_kp(x, y, sbits):
    x = np.uint64(np.uint16(x))
    y = np.uint64(np.uint16(y))
    s = np.uint64(np.uint32(sbits))
    return np.uint64(x | (y << np.uint64(16)) | (s << np.uint64(32)))

def u32_to_f_scalar(u32_val):
    # exact reinterpretation uint32 -> float32 (like union in C++)
    return np.array([np.uint32(u32_val)], dtype=np.uint32).view(np.float32)[0]

def u32_array_to_f32(arr_u32):
    arr_u32 = np.asarray(arr_u32, dtype=np.uint32)
    return arr_u32.view(np.float32)

In [ ]:
nms_count = read_int_txt(nms_count_txt)
nms_kps = read_u64_bin(nms_kps_bin, nms_count)

print("nms_count (txt) =", nms_count)
print("nms_kps loaded  =", len(nms_kps))

assert len(nms_kps) == nms_count, "Mismatch between count file and .bin length"

# quick peek
for i in range(min(5, len(nms_kps))):
    w = nms_kps[i]
    print(i, "x=", int(kp_x(w)), "y=", int(kp_y(w)), "sbits=", int(kp_sbits(w)))

In [ ]:
def select_topK(in_kps_u64, K):
    """
    select_topK:
      - initialize first min(in_count, K)
      - find min score among current topK
      - scan rest, replace current min only when s > min_s
      - recompute min after each replacement
    Returns xs, ys, ss arrays (length K), and n (=min(in_count,K))
    """
    in_kps_u64 = np.asarray(in_kps_u64, dtype=np.uint64)
    in_count = int(len(in_kps_u64))
    n = min(in_count, K)

    xs = np.zeros(K, dtype=np.uint16)
    ys = np.zeros(K, dtype=np.uint16)
    ss = np.zeros(K, dtype=np.uint32)

    # InitFill
    for i in range(K):
        if i < n:
            w = in_kps_u64[i]
            xs[i] = kp_x(w)
            ys[i] = kp_y(w)
            ss[i] = kp_sbits(w)
        else:
            xs[i] = np.uint16(0)
            ys[i] = np.uint16(0)
            ss[i] = np.uint32(0)

    if n == 0:
        return xs, ys, ss, 0

    # FindMin0 over first n valid entries
    min_idx = 0
    min_s = ss[0]
    for i in range(1, n):
        # note: strict '<' to keep first minimum index 
        if ss[i] < min_s:
            min_s = ss[i]
            min_idx = i

    # ScanRest
    for i in range(n, in_count):
        w = in_kps_u64[i]
        s = kp_sbits(w)
        # note: strict '>' 
        if s > min_s:
            xs[min_idx] = kp_x(w)
            ys[min_idx] = kp_y(w)
            ss[min_idx] = s

            # Recompute min over full K (HLS loops t=1..K-1)
            min_idx = 0
            min_s = ss[0]
            for t in range(1, K):
                if ss[t] < min_s:
                    min_s = ss[t]
                    min_idx = t

    return xs, ys, ss, n

In [ ]:
def radix_sort_desc(xs, ys, ss, K):
    """
    radix_sort_desc on score bits (uint32), descending.
    Operates on arrays of length K.
    """
    xs = np.asarray(xs, dtype=np.uint16).copy()
    ys = np.asarray(ys, dtype=np.uint16).copy()
    ss = np.asarray(ss, dtype=np.uint32).copy()

    tx = np.zeros(K, dtype=np.uint16)
    ty = np.zeros(K, dtype=np.uint16)
    ts = np.zeros(K, dtype=np.uint32)

    for p in range(4):
        shift = p * 8

        count = np.zeros(256, dtype=np.uint16)
        pos   = np.zeros(256, dtype=np.uint16)

        # histogram
        for i in range(K):
            key = np.uint8((int(ss[i]) >> shift) & 0xFF)
            count[key] = np.uint16(count[key] + 1)

        # prefix sums for DESCENDING order (255..0)
        running = np.uint16(0)
        for b in range(255, -1, -1):
            pos[b] = running
            running = np.uint16(running + count[b])

        # scatter (stable w.r.t. current order)
        for i in range(K):
            key = np.uint8((int(ss[i]) >> shift) & 0xFF)
            pidx = int(pos[key])
            pos[key] = np.uint16(pos[key] + 1)

            tx[pidx] = xs[i]
            ty[pidx] = ys[i]
            ts[pidx] = ss[i]

        # copy back
        xs[:] = tx
        ys[:] = ty
        ss[:] = ts

    return xs, ys, ss

In [ ]:
def anms_core(xs, ys, ss, K, N, c):
    """
      1) radix sort desc by score bits
      2) convert score bits to float32
      3) compute radius^2 using stronger points (j < i and sf[j] > c*sf[i])
      4) keep top-N by largest radius^2 using replacement-by-min
      5) output packed kp in current output-array order (not resorted)
    Returns np.uint64 array of length 'filled'
    """
    INF = np.uint32(0xFFFFFFFF)

    # 1) sort descending by score bits
    xs, ys, ss = radix_sort_desc(xs, ys, ss, K)

    # 2) convert scores to float32 (reinterpret bits)
    sf = u32_array_to_f32(ss.copy())

    # 3) suppression radius^2
    r2 = np.zeros(K, dtype=np.uint32)

    for i in range(K):
        best = INF
        thr = np.float32(c) * np.float32(sf[i])

        for j in range(i):
            # strict '>' matches HLS
            if np.float32(sf[j]) > thr:
                dx = int(xs[i]) - int(xs[j])
                dy = int(ys[i]) - int(ys[j])
                d2 = dx*dx + dy*dy
                # cast to uint32 like HW
                d2u = np.uint32(d2 & 0xFFFFFFFF)
                if d2u < best:
                    best = d2u
        r2[i] = best

    # 4) keep top-N by largest radius^2
    ox  = np.zeros(N, dtype=np.uint16)
    oy  = np.zeros(N, dtype=np.uint16)
    os_ = np.zeros(N, dtype=np.uint32)
    or2 = np.zeros(N, dtype=np.uint32)

    filled = 0

    for i in range(K):
        val = r2[i]

        if filled < N:
            ox[filled]  = xs[i]
            oy[filled]  = ys[i]
            os_[filled] = ss[i]
            or2[filled] = val
            filled += 1
        else:
            # find minimum radius in current output (strict '<' keeps first min)
            min_i = 0
            minv = or2[0]
            for t in range(1, N):
                if or2[t] < minv:
                    minv = or2[t]
                    min_i = t

            # strict '>' 
            if val > minv:
                ox[min_i]  = xs[i]
                oy[min_i]  = ys[i]
                os_[min_i] = ss[i]
                or2[min_i] = val

    # 5) pack output in current output-array order
    out = np.zeros(filled, dtype=np.uint64)
    for i in range(filled):
        out[i] = pack_kp(ox[i], oy[i], os_[i])

    return out

In [ ]:
def anms(nms_kps_u64, nms_count, K_top, N_final, c, max_in):
    """
    Returns (out_kps_u64, out_count)
    """
    KMAX = 2048
    NMAX = 1024

    # parameter checks
    if K_top <= 0 or N_final <= 0 or K_top > KMAX or N_final > NMAX or N_final > K_top:
        return np.zeros(0, dtype=np.uint64), 0

    in_count = int(nms_count)
    if in_count < 0:
        in_count = 0
    if in_count > int(max_in):
        in_count = int(max_in)

    if in_count == 0:
        return np.zeros(0, dtype=np.uint64), 0

    nms_kps_u64 = np.asarray(nms_kps_u64, dtype=np.uint64)[:in_count]

    Kuse = min(in_count, K_top)

    # allocate KMAX-like arrays conceptually, but we only need K_top slots
    if in_count <= K_top:
        xs = np.zeros(K_top, dtype=np.uint16)
        ys = np.zeros(K_top, dtype=np.uint16)
        ss = np.zeros(K_top, dtype=np.uint32)

        # CopyAll
        for i in range(Kuse):
            w = nms_kps_u64[i]
            xs[i] = kp_x(w)
            ys[i] = kp_y(w)
            ss[i] = kp_sbits(w)

        # Pad already zero
    else:
        xs, ys, ss, _n = select_topK(nms_kps_u64, K_top)

    out = anms_core(xs, ys, ss, Kuse, N_final, c)
    return out, int(len(out))

In [ ]:
max_in = nms_count  

py_anms_kps, py_anms_count = anms(
    nms_kps_u64=nms_kps,
    nms_count=nms_count,
    K_top=K_top,
    N_final=N_final,
    c=c_anms,
    max_in=max_in
)

print("Python ANMS output count =", py_anms_count)
assert py_anms_count == 1000, f"Expected 1000, got {py_anms_count}"
print("len(output) == 1000")

In [ ]:
write_u64_bin(py_anms_kps_bin, py_anms_kps)
write_int_txt(py_anms_count_txt, py_anms_count)

print("Saved:", py_anms_kps_bin)
print("Saved:", py_anms_count_txt)

In [ ]:
hw_anms_count = read_int_txt(hw_anms_count_txt)
hw_anms_kps = read_u64_bin(hw_anms_kps_bin, hw_anms_count)

print("HW ANMS count =", hw_anms_count)
print("HW ANMS len   =", len(hw_anms_kps))
print("PY ANMS count =", py_anms_count)

assert hw_anms_count == len(hw_anms_kps)
assert py_anms_count == len(py_anms_kps)

In [ ]:
same_count = (len(hw_anms_kps) == len(py_anms_kps))
same_all = same_count and np.array_equal(hw_anms_kps, py_anms_kps)

print("same_count:", same_count)
print("bit-exact equal:", same_all)

if not same_all:
    n = min(len(hw_anms_kps), len(py_anms_kps))
    mismatch_idx = None
    for i in range(n):
        if hw_anms_kps[i] != py_anms_kps[i]:
            mismatch_idx = i
            break

    if mismatch_idx is None and len(hw_anms_kps) != len(py_anms_kps):
        print("Counts differ but common prefix is equal.")
    else:
        i = mismatch_idx
        wh = hw_anms_kps[i]
        wp = py_anms_kps[i]
        print(f"First mismatch at index {i}")
        print("HW raw:", int(wh))
        print("PY raw:", int(wp))
        print("HW decoded:", int(kp_x(wh)), int(kp_y(wh)), int(kp_sbits(wh)))
        print("PY decoded:", int(kp_x(wp)), int(kp_y(wp)), int(kp_sbits(wp)))
else:
    print("Python ANMS matches HW ANMS bit-exact")

In [ ]:
cpu_anms_kps_bin   = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_cpu_anms_kps.bin"
cpu_anms_count_txt = BASE_DIR / "seq1_img1_K2048_N1000_c0.90_cpu_anms_count.txt"
#Read CPU ANMS output
cpu_anms_count = read_int_txt(cpu_anms_count_txt)
cpu_anms_kps = read_u64_bin(cpu_anms_kps_bin, cpu_anms_count)

print("CPU ANMS count =", cpu_anms_count)
print("CPU ANMS len   =", len(cpu_anms_kps))

assert cpu_anms_count == len(cpu_anms_kps)
assert cpu_anms_count == 1000, f"Expected CPU output count 1000, got {cpu_anms_count}"

#  CPU vs HW
cpu_vs_hw_same_count = (len(cpu_anms_kps) == len(hw_anms_kps))
cpu_vs_hw_same_all = cpu_vs_hw_same_count and np.array_equal(cpu_anms_kps, hw_anms_kps)

print("CPU vs HW same_count:", cpu_vs_hw_same_count)
print("CPU vs HW bit-exact :", cpu_vs_hw_same_all)

assert cpu_vs_hw_same_all, "CPU ANMS does not match HW ANMS bit-exact"

#  CPU vs PY-
cpu_vs_py_same_count = (len(cpu_anms_kps) == len(py_anms_kps))
cpu_vs_py_same_all = cpu_vs_py_same_count and np.array_equal(cpu_anms_kps, py_anms_kps)

print("CPU vs PY same_count:", cpu_vs_py_same_count)
print("CPU vs PY bit-exact :", cpu_vs_py_same_all)

assert cpu_vs_py_same_all, "CPU ANMS does not match Python ANMS bit-exact"

In [ ]:
print("=== ANMS Validation Summary (seq1_img1) ===")
print(f"NMS input count : {nms_count}")
print(f"K_top           : {K_top}")
print(f"N_final         : {N_final}")
print(f"c               : {c_anms}")

print(f"HW ANMS count   : {hw_anms_count}")
print(f"PY ANMS count   : {py_anms_count}")
print(f"CPU ANMS count  : {cpu_anms_count}")

print("PY  vs HW       : bit-exact PASS")
print("CPU vs HW       : bit-exact PASS")
print("CPU vs PY       : bit-exact PASS")

print("\nConclusion: Python, CPU C++, and HW ANMS match bit-exact on the same frozen NMS input.")